<a href="https://colab.research.google.com/github/FIGARO79/GanaBaloto/blob/main/An%C3%A1lisis_Baloto_con_Refactorizaci%C3%B3n_JAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

if not os.path.exists('/content/drive/MyDrive/GanaBaloto'):
  print("Installing and Running For First Time.")
  %cd /content/drive/MyDrive
  !git clone https://github.com/FIGARO79/GanaBaloto.git
  !cp -r /content/drive/MyDrive/GanaBaloto
else:
  print("Repository already exist. Continue...")
  %cd /content/drive/MyDrive/GanaBaloto


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exist. Continue...
/content/drive/MyDrive/GanaBaloto


In [ ]:
# Importar librerías necesarias
import pandas as pd
from IPython.display import display, HTML
import random
from itertools import combinations
import jax
import jax.numpy as jnp

# --- Configuración ---
# IMPORTANTE: Actualiza esta ruta a la ubicación correcta de tu archivo Excel en Google Drive
# Asegúrate de que tu Google Drive esté montado en Colab (normalmente en /content/drive)
wb = '/content/drive/MyDrive/GanaBaloto/baloto.xlsx'
ws = 'Baloto'
columns_to_analyze = ['B1', 'B2', 'B3', 'B4', 'B5']
super_balota_column = 'SB'
prize_column = 'Premios 5+1'
num_combinations_to_generate = 10

# --- Carga y Preprocesamiento de Datos ---
try:
    # Leer el archivo Excel
    sheet_data = pd.read_excel(wb, sheet_name=ws)

    # Eliminar filas con valores faltantes de Super Balota y convertir a entero
    sheet_data = sheet_data.dropna(subset=[super_balota_column])
    sheet_data[super_balota_column] = sheet_data[super_balota_column].astype(int)

    # Convertir columnas de balotas principales a numérico, forzando errores a NaN
    for col in columns_to_analyze:
        sheet_data[col] = pd.to_numeric(sheet_data[col], errors='coerce')
        # Eliminar filas donde falló la conversión
        sheet_data = sheet_data.dropna(subset=[col])
        # Convertir las válidas a entero
        sheet_data[col] = sheet_data[col].astype(int)

    print("Datos cargados y preprocesados exitosamente.")

except FileNotFoundError:
    print(f"Error: Archivo no encontrado en {wb}. Por favor, asegúrate de que la ruta sea correcta y Google Drive esté montado.")
    # Salir o manejar el error apropiadamente si no se encuentra el archivo
    exit()
except KeyError as e:
    print(f"Error: Columna {e} no encontrada en la hoja '{ws}'. Por favor, revisa los nombres de las columnas.")
    exit()
except Exception as e:
    print(f"Ocurrió un error inesperado durante la carga de datos: {e}")
    exit()


# --- Análisis de Sorteos Duplicados ---
print("\n--- Analizando Sorteos Duplicados ---")
combinations_df = sheet_data[columns_to_analyze + [super_balota_column]]
# Usar .duplicated() en el subconjunto relevante
duplicate_mask = combinations_df.duplicated(keep=False)
duplicate_combinations = sheet_data[duplicate_mask]

if duplicate_combinations.empty:
    print("No se encontraron combinaciones duplicadas.")
else:
    print("Combinaciones Duplicadas Encontradas:")
    display(duplicate_combinations)


# --- Análisis de Frecuencia ---
print("\n--- Calculando Frecuencias de Números ---")

# Calcular frecuencias para todos los sorteos
all_draws_frequencies = {}
for column in columns_to_analyze:
    # Obtener los 5 números más frecuentes para cada columna de balota
    all_draws_frequencies[column] = sheet_data[column].value_counts().head(5).index.tolist()
# Obtener los 5 números más frecuentes para Super Balota
all_draws_frequencies["Super Balota"] = sheet_data[super_balota_column].value_counts().head(5).index.tolist()

# Filtrar datos para sorteos donde se ganó el premio mayor ('Premios 5+1' > 0)
filtered_data = sheet_data[sheet_data[prize_column] > 0].copy() # Usar .copy() para evitar SettingWithCopyWarning

# Calcular frecuencias para sorteos filtrados (ganadores)
refined_predictions = {}
if not filtered_data.empty:
    for column in columns_to_analyze:
        refined_predictions[column] = filtered_data[column].value_counts().head(5).index.tolist()
    refined_predictions["Super Balota"] = filtered_data[super_balota_column].value_counts().head(5).index.tolist()
    print("Frecuencias calculadas para todos los sorteos y sorteos ganadores.")
else:
    print("No se encontraron sorteos con ganadores del premio mayor. Las predicciones refinadas estarán vacías.")
    # Manejar el caso donde refined_predictions estaría vacío
    for column in columns_to_analyze:
        refined_predictions[column] = []
    refined_predictions["Super Balota"] = []


# Crear DataFrames para mostrar frecuencias
df_all_draws = pd.DataFrame.from_dict(all_draws_frequencies, orient='index')
df_all_draws = df_all_draws.rename(columns={i: f'Top {i+1}' for i in range(df_all_draws.shape[1])})

df_refined_predictions = pd.DataFrame.from_dict(refined_predictions, orient='index')
df_refined_predictions = df_refined_predictions.rename(columns={i: f'Top {i+1}' for i in range(df_refined_predictions.shape[1])})


# --- Cálculo de Puntaje Basado en JAX ---
# Definir la función para calcular el puntaje de frecuencia usando JAX
# Nota: Esto calcula un puntaje basado en frecuencia histórica, no una probabilidad matemática estricta.
# Aplicamos jax.jit para compilar la función para posible aceleración si se llama repetidamente.
@jax.jit
def calculate_frequency_score_jax(combination_jax, sb_jax, b1_col_jax, b2_col_jax, b3_col_jax, b4_col_jax, b5_col_jax, sb_col_jax):
    """
    Calcula un puntaje de frecuencia para una combinación dada usando JAX.

    Args:
        combination_jax (jax.Array): Arreglo JAX de los 5 números principales.
        sb_jax (jax.Array): Arreglo JAX conteniendo el número de Super Balota.
        b1_col_jax a b5_col_jax (jax.Array): Arreglos JAX para cada columna histórica de balota.
        sb_col_jax (jax.Array): Arreglo JAX para la columna histórica de Super Balota.

    Returns:
        jax.Array: El puntaje de frecuencia calculado.
    """
    # Obtener total de sorteos de la longitud de una columna
    total_draws = b1_col_jax.shape[0]
    # Usar flotante para el puntaje
    score = 0.0

    # Calcular contribución al puntaje de las balotas principales
    ball_columns_jax = [b1_col_jax, b2_col_jax, b3_col_jax, b4_col_jax, b5_col_jax]
    for i, number in enumerate(combination_jax):
        # Contar frecuencia del número en la columna histórica correspondiente
        frequency = jnp.sum(ball_columns_jax[i] == number)
        # Añadir la frecuencia relativa al puntaje
        score += frequency / total_draws

    # Calcular contribución al puntaje de la Super Balota
    sb_frequency = jnp.sum(sb_col_jax == sb_jax)
    score += sb_frequency / total_draws

    return score

# Preparar arreglos JAX de los datos históricos UNA VEZ para eficiencia
# Convertir columnas pandas relevantes a arreglos JAX
b_cols_jax = [jnp.array(sheet_data[f'B{i+1}'].values) for i in range(5)]
sb_col_jax = jnp.array(sheet_data[super_balota_column].values)


# --- Generación de Combinaciones ---
def generate_probable_combinations(predictions, num_combinations=10):
    """
    Genera combinaciones basadas en los números más frecuentes de las predicciones
    y calcula sus puntajes de frecuencia usando la función JAX.

    Args:
        predictions (dict): Diccionario conteniendo listas de números frecuentes para cada balota y SB.
                           Se esperan las claves: 'B1'...'B5', 'Super Balota'.
        num_combinations (int): Número de combinaciones a generar.

    Returns:
        list: Una lista de tuplas, donde cada tupla es (lista_combinacion, numero_sb, puntaje).
              Retorna una lista vacía si las predicciones son insuficientes.
    """
    # Verificar si el diccionario de predicciones tiene las claves requeridas y listas no vacías
    required_keys = columns_to_analyze + ["Super Balota"]
    if not all(key in predictions and predictions[key] for key in required_keys):
         print("Advertencia: Datos insuficientes en 'predictions' para generar combinaciones.")
         # Retornar lista vacía si faltan datos
         return []

    # Aplanar las listas de números frecuentes para las balotas principales
    all_probable_main_numbers = [num for key in columns_to_analyze if key in predictions for num in predictions[key]]
    # Asegurar números únicos intentando preservar algo de orden
    unique_probable_main_numbers = sorted(list(dict.fromkeys(all_probable_main_numbers)))

    # Obtener la lista de números frecuentes de Super Balota
    probable_sb_numbers = predictions.get("Super Balota", [])

    # Asegurar que tenemos suficientes números únicos para muestrear
    if len(unique_probable_main_numbers) < 5 or not probable_sb_numbers:
        print("Advertencia: No hay suficientes números únicos frecuentes para generar combinaciones.")
        return []

    generated_combinations = []
     # Limitar intentos para evitar bucles infinitos si el muestreo es difícil
    attempts = 0
    # Permitir más intentos de los necesarios
    max_attempts = num_combinations * 5

    while len(generated_combinations) < num_combinations and attempts < max_attempts:
        attempts += 1
        try:
            # Muestrear 5 números únicos para la combinación principal
            combination = random.sample(unique_probable_main_numbers, 5)
            # Muestrear 1 número para la Super Balota
            sb = random.choice(probable_sb_numbers)

            # Convertir combination y sb a arreglos JAX para la función de cálculo
            combination_jax = jnp.array(combination)
             # Pasar SB como arreglo JAX (incluso si es un solo elemento)
            sb_jax = jnp.array(sb)

            # Calcular el puntaje de frecuencia usando la función JAX
            # Pasar las columnas de datos históricos pre-convertidas
            score = calculate_frequency_score_jax(combination_jax, sb_jax, *b_cols_jax, sb_col_jax)

            # Convertir puntaje de arreglo JAX a float de Python para almacenamiento/visualización
            score_float = float(score)

            # Añadir el resultado (combinación como lista, sb como int, puntaje como float)
            generated_combinations.append((sorted(combination), sb, score_float))

        except ValueError as e:
            # Manejar errores potenciales durante el muestreo (ej., k > población)
            print(f"Error de muestreo: {e}. Omitiendo este intento.")
        except Exception as e:
            # Capturar cualquier otro error inesperado
            print(f"Ocurrió un error inesperado durante la generación de combinaciones: {e}")

    if attempts >= max_attempts and len(generated_combinations) < num_combinations:
        print(f"Advertencia: Solo se pudieron generar {len(generated_combinations)} combinaciones después de {max_attempts} intentos.")

    # Ordenar combinaciones por puntaje en orden descendente (puntaje más alto primero)
    generated_combinations.sort(key=lambda x: x[2], reverse=True)

    return generated_combinations


# --- Entrada de Jugada Manual ---
def jugada_manual():
    """
    Permite al usuario ingresar una combinación manual y calcula su puntaje de frecuencia.

    Returns:
        tuple: (lista_combinacion, numero_sb, puntaje_float) o None si la entrada es inválida.
    """
    while True:
        try:
            combinacion = []
            print("\n--- Ingreso de Jugada Manual ---")
            # Ingresar números principales
            for i in range(5):
                while True:
                    try:
                        number_str = input(f"Ingrese el número {i + 1} de la combinación (1-43): \n")
                        number = int(number_str)
                        if 1 <= number <= 43:
                            if number not in combinacion:
                                combinacion.append(number)
                                # Salir del bucle interno para este número
                                break
                            else:
                                print("Error: Número repetido. Ingrese un número único.")
                        else:
                            print("Error: Número fuera del rango (1-43).")
                    except ValueError:
                        print("Error: Entrada inválida. Por favor, ingrese un número entero.")

            # Ingresar Super Balota
            while True:
                 try:
                    sb_str = input("Ingrese el número de la Super Balota (1-16): \n")
                    sb = int(sb_str)
                    if 1 <= sb <= 16:
                         # Salir del bucle interno para Super Balota
                        break
                    else:
                        print("Error: Número fuera del rango (1-16).")
                 except ValueError:
                    print("Error: Entrada inválida. Por favor, ingrese un número entero.")

            # Calcular puntaje usando función JAX
            combination_jax = jnp.array(combinacion)
            sb_jax = jnp.array(sb)
            score = calculate_frequency_score_jax(combination_jax, sb_jax, *b_cols_jax, sb_col_jax)
             # Convertir a flotante
            score_float = float(score)

            print(f"\nCombinación ingresada: {sorted(combinacion)}, Super Balota: {sb}")
            # Mostrar puntaje - Nota: Un puntaje más alto significa que los números aparecieron históricamente con más frecuencia
            # No es un porcentaje de probabilidad directo.
            print(f"Puntaje de Frecuencia Histórica: {score_float:.4f}") # Mostrar puntaje con más precisión

            return sorted(combinacion), sb, score_float

        except Exception as e: # Capturar errores potenciales durante el proceso
            print(f"Ocurrió un error durante el ingreso manual: {e}")
             # Indicar fallo
            return None


# --- Lógica Principal de Ejecución ---

# Mostrar Tablas de Frecuencia
print("\n--- Mostrando Tablas de Frecuencia ---")
html_str_freq = f"""
<h2 style='text-align:center;'>Análisis de Frecuencia de Números Baloto</h2>
<table style='width:100%; border-collapse: collapse;'>
  <tr>
    <td style='vertical-align: top; padding: 10px; border: 1px solid #ddd;'>
      <h3>Números más frecuentes (Todos los Sorteos)</h3>
      {df_all_draws.to_html(classes='table table-striped', justify='center')}
    </td>
    <td style='vertical-align: top; padding: 10px; border: 1px solid #ddd;'>
      <h3>Números más frecuentes (Sorteos con Premio 5+1)</h3>
      {df_refined_predictions.to_html(classes='table table-striped', justify='center')}
    </td>
  </tr>
</table>
"""
display(HTML(html_str_freq))

# Bucle de Juego Manual
while True:
    response = input("\n¿Desea ingresar una jugada manual? (s/n): \n").lower()
    if response == 's':
        manual_result = jugada_manual()
        if manual_result:
            # Optionally store or use the manual result
            pass
    elif response == 'n':
        print("\nProcediendo a generar jugadas automáticas.")
        break
    else:
        print("Respuesta no válida. Por favor, ingrese 's' o 'n'.")

# Bucle de Generación y Visualización Automática de Combinaciones
print("\n--- Generando Combinaciones Automáticas (Basadas en Sorteos Ganadores) ---")

while True:
    # Generar combinaciones usando predicciones refinadas (de sorteos ganadores)
    # Si refined_predictions está vacío, generate_probable_combinations lo manejará
    probable_combinations = generate_probable_combinations(refined_predictions, num_combinations=num_combinations_to_generate)

    if not probable_combinations:
        print("No se pudieron generar combinaciones automáticas (datos insuficientes o error).")
        # Decidir si quieres intentar con all_draws_frequencies o parar
        print("Intentando generar combinaciones basadas en TODOS los sorteos...")
        probable_combinations = generate_probable_combinations(all_draws_frequencies, num_combinations=num_combinations_to_generate)
        if not probable_combinations:
             print("Tampoco se pudieron generar combinaciones con todos los sorteos. Finalizando.")
              # Salir del bucle si la generación falla completamente
             break


    # Crear DataFrame para visualización
    data_for_df = []
    for combinacion, sb, score in probable_combinations:
        # Formatear lista de combinación como cadena
        combinacion_str = ', '.join(map(str, combinacion))
        # Añadir fila de datos: cadena de combinación, número SB, puntaje
        data_for_df.append([combinacion_str, sb, score])

    # Crear el DataFrame de pandas
    df_combinations = pd.DataFrame(data_for_df, columns=['Combinacion', 'Super Balota', 'Puntaje Frecuencia'])
    # Establecer índice comenzando desde 1
    df_combinations.index = range(1, len(df_combinations) + 1)

    # Formatear la columna 'Puntaje Frecuencia' para visualización (ej., 4 decimales)
    # HACER ESTE PASO EN EL DATAFRAME DE PANDAS
    df_combinations['Puntaje Frecuencia'] = df_combinations['Puntaje Frecuencia'].map('{:.4f}'.format)


    # Mostrar el DataFrame como HTML
    html_str_comb = f"""
    <div style='margin-top: 20px;'>
      <h3 style='text-align:center;'>Combinaciones Automáticas Sugeridas y su Puntaje de Frecuencia</h3>
      {df_combinations.to_html(classes='table table-hover', justify='center', index=True)}
    </div>
    """
    display(HTML(html_str_comb))

    # Preguntar al usuario si quiere más combinaciones
    response = input("\n¿Desea generar más combinaciones automáticas? (s/n): \n").lower()
    if response == 's':
         # Continuar el bucle para generar más
        continue
    elif response == 'n':
        print("\n¡Mucha suerte con tus números!")
         # Salir del bucle
        break
    elif response != 's' or response != 'n':
        print("Respuesta no válida. Por favor, ingrese 's' o 'n'.")
        continue
    else:
        print("Respuesta no válida. Finalizando.")
         # Salir con entrada inválida después de la generación
        break

print("\n--- Análisis Finalizado ---")